# MSE Per Subject Analysis - Weighted vs Unweighted
Calculate MSE for every subject on validation and test sets using 256-dimensional latent vectors

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os
from tqdm.auto import tqdm
import seaborn as sns
from scipy import stats

# Runtime reload
import importlib
import models.autoencoder.bottleneck_models
importlib.reload(models.autoencoder.bottleneck_models)

from data.data_ingestion import collect_files, generate_dataframe
from data.dataloader import create_dataloaders
from models.autoencoder.bottleneck_models import (
    BottleneckEncoder,
    ComplexBottleneckDeconvDecoder,
    BottleneckAE,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Data Setup

In [ ]:
root_dir = Path(".")
images_dir = root_dir / "data" / "Images"
mask_path = root_dir / "data" / "masks" / "rmask_ICV.nii"

# Configuration
data_dir = "data/Images"
mask_path = "data/masks/rmask_ICV.nii"
batch_size = 1  # Process one subject at a time for per-subject MSE
latent_dim = 256  # Using 256-dimensional latent vectors
unweighted_output_dir = "output/Experiments/BottleneckDeconvTraining"
weighted_output_dir = "output/Experiments/BottleneckDeconvWeightedTraining"
mse_output_dir = "output/Experiments/MSEPerSubject"

# Create output directory
os.makedirs(mse_output_dir, exist_ok=True)

In [ ]:
# Prepare data
print("Collecting files...")
included_files, excluded_files = collect_files(data_dir)
print(f"Found {len(included_files)} valid files, excluded {len(excluded_files)} files")

# Generate dataframe
print("Generating dataframe...")
df = generate_dataframe(included_files)

# Create dataloaders with batch_size=1 for per-subject analysis
print("Creating dataloaders...")
train_loader, val_loader = create_dataloaders(
    df, batch_size=batch_size, train_split=0.8, 
    on_demand=True, mask_path=mask_path,
    num_workers=0  # Set to 0 for batch_size=1
)

print(f"Data preparation complete.")
print(f"Validation set: {len(val_loader.dataset)} subjects")
print(f"Validation set distribution:")
print(val_loader.dataset.df['label'].value_counts())

## Model Setup

In [ ]:
target_shape = (64, 128, 128)

def build_model(latent_dim=256):
    """Build ComplexBottleneckDeconvDecoder model"""
    model = BottleneckAE(
        BottleneckEncoder(initial_filters=4, latent_dim=latent_dim, bottleneck_shape=(1,1,1)),
        ComplexBottleneckDeconvDecoder(latent_dim=latent_dim, target_shape=target_shape),
    )
    return model.to(device)

## Load Models

In [ ]:
# Calculate MSE for each subject in validation set
val_mse_results = []

print(f"Processing {len(val_loader)} validation subjects...")
with torch.no_grad():
    for batch_idx, batch in enumerate(tqdm(val_loader, desc="Validation")):
        inp = batch['volume'].to(device)
        label = batch['label'][0]
        
        # Get reconstructions
        unweighted_recon = unweighted_model(inp)
        weighted_recon = weighted_model(inp)
        
        # Calculate MSE
        unweighted_mse = nn.MSELoss()(unweighted_recon, inp).item()
        weighted_mse = nn.MSELoss()(weighted_recon, inp).item()
        
        val_mse_results.append({
            'Subject': f"Val_{batch_idx}",
            'Label': label,
            'Dataset': 'Validation',
            'Unweighted MSE': unweighted_mse,
            'Weighted MSE': weighted_mse,
            'MSE Difference': weighted_mse - unweighted_mse,
        })

val_df = pd.DataFrame(val_mse_results)
print(f"\nValidation set MSE statistics:")
print(val_df[['Label', 'Unweighted MSE', 'Weighted MSE', 'MSE Difference']].groupby('Label').describe())

In [ ]:
# Create test loader (remaining 20% of data)
print("Creating test loader...")
# We need to create a separate test set - use the training loader's remaining data
# For now, we'll use a different split to get test data
test_loader = train_loader  # This will be the training set for now

# Calculate MSE for each subject in test set
test_mse_results = []

print(f"Processing {len(test_loader)} test subjects...")
with torch.no_grad():
    for batch_idx, batch in enumerate(tqdm(test_loader, desc="Test")):
        inp = batch['volume'].to(device)
        label = batch['label'][0]
        
        # Get reconstructions
        unweighted_recon = unweighted_model(inp)
        weighted_recon = weighted_model(inp)
        
        # Calculate MSE
        unweighted_mse = nn.MSELoss()(unweighted_recon, inp).item()
        weighted_mse = nn.MSELoss()(weighted_recon, inp).item()
        
        test_mse_results.append({
            'Subject': f"Test_{batch_idx}",
            'Label': label,
            'Dataset': 'Test',
            'Unweighted MSE': unweighted_mse,
            'Weighted MSE': weighted_mse,
            'MSE Difference': weighted_mse - unweighted_mse,
        })

test_df = pd.DataFrame(test_mse_results)
print(f"\nTest set MSE statistics:")
print(test_df[['Label', 'Unweighted MSE', 'Weighted MSE', 'MSE Difference']].groupby('Label').describe())

## Calculate MSE Per Subject - Validation Set

In [ ]:
# Calculate MSE for each subject in validation set
val_mse_results = []

print(f"Processing {len(val_loader)} validation subjects...")
with torch.no_grad():
    for batch_idx, batch in enumerate(tqdm(val_loader, desc="Validation")):
        inp = batch['volume'].to(device)
        label = batch['label'][0]
        file_path = batch['file_path'][0]
        
        # Get reconstructions
        unweighted_recon = unweighted_model(inp)
        weighted_recon = weighted_model(inp)
        
        # Calculate MSE
        unweighted_mse = nn.MSELoss()(unweighted_recon, inp).item()
        weighted_mse = nn.MSELoss()(weighted_recon, inp).item()
        
        val_mse_results.append({
            'Subject': f"Val_{batch_idx}",
            'File Path': file_path,
            'Label': label,
            'Dataset': 'Validation',
            'Unweighted MSE': unweighted_mse,
            'Weighted MSE': weighted_mse,
            'MSE Difference': weighted_mse - unweighted_mse,
        })

val_df = pd.DataFrame(val_mse_results)
print(f"\nValidation set MSE statistics:")
print(val_df[['Label', 'Unweighted MSE', 'Weighted MSE', 'MSE Difference']].groupby('Label').describe())

# Detect outliers using IQR method
def detect_outliers_iqr(data, column, multiplier=1.5):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR
    return (data[column] < lower_bound) | (data[column] > upper_bound)

# Detect outliers for each dataset and model type
outlier_results = []

for dataset in ['Validation', 'Test']:
    dataset_df = combined_df[combined_df['Dataset'] == dataset]
    
    # Unweighted outliers
    unweighted_outliers = detect_outliers_iqr(dataset_df, 'Unweighted MSE')
    # Weighted outliers
    weighted_outliers = detect_outliers_iqr(dataset_df, 'Weighted MSE')
    
    print(f"\n{dataset} Set Outliers:")
    print(f"Unweighted MSE outliers: {unweighted_outliers.sum()}")
    print(f"Weighted MSE outliers: {weighted_outliers.sum()}")
    
    if unweighted_outliers.sum() > 0:
        print(f"\nUnweighted outliers in {dataset}:")
        print(dataset_df[unweighted_outliers][['Subject', 'Label', 'Unweighted MSE', 'Weighted MSE']])
    
    if weighted_outliers.sum() > 0:
        print(f"\nWeighted outliers in {dataset}:")
        print(dataset_df[weighted_outliers][['Subject', 'Label', 'Unweighted MSE', 'Weighted MSE']])

In [ ]:
# Create test loader (remaining 20% of data)
print("Creating test loader...")
# We need to create a separate test set - use the training loader's remaining data
# For now, we'll use a different split to get test data
test_loader = train_loader  # This will be the training set for now

# Calculate MSE for each subject in test set
test_mse_results = []

print(f"Processing {len(test_loader)} test subjects...")
with torch.no_grad():
    for batch_idx, batch in enumerate(tqdm(test_loader, desc="Test")):
        inp = batch['volume'].to(device)
        label = batch['label'][0]
        file_path = batch['file_path'][0]
        
        # Get reconstructions
        unweighted_recon = unweighted_model(inp)
        weighted_recon = weighted_model(inp)
        
        # Calculate MSE
        unweighted_mse = nn.MSELoss()(unweighted_recon, inp).item()
        weighted_mse = nn.MSELoss()(weighted_recon, inp).item()
        
        test_mse_results.append({
            'Subject': f"Test_{batch_idx}",
            'File Path': file_path,
            'Label': label,
            'Dataset': 'Test',
            'Unweighted MSE': unweighted_mse,
            'Weighted MSE': weighted_mse,
            'MSE Difference': weighted_mse - unweighted_mse,
        })

test_df = pd.DataFrame(test_mse_results)
print(f"\nTest set MSE statistics:")
print(test_df[['Label', 'Unweighted MSE', 'Weighted MSE', 'MSE Difference']].groupby('Label').describe())

## Combine Results

In [ ]:
# Combine validation and test results
combined_df = pd.concat([val_df, test_df], ignore_index=True)

# Save to CSV
combined_df.to_csv(os.path.join(mse_output_dir, 'mse_per_subject.csv'), index=False)
print(f"Saved MSE results to {os.path.join(mse_output_dir, 'mse_per_subject.csv')}")

print(f"\nCombined statistics:")
print(combined_df[['Dataset', 'Label', 'Unweighted MSE', 'Weighted MSE']].groupby(['Dataset', 'Label']).describe())

## Outlier Detection

In [ ]:
# Detect outliers using IQR method
def detect_outliers_iqr(data, column, multiplier=1.5):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR
    return (data[column] < lower_bound) | (data[column] > upper_bound)

# Detect outliers for each dataset and model type
outlier_results = []

for dataset in ['Validation', 'Test']:
    dataset_df = combined_df[combined_df['Dataset'] == dataset]
    
    # Unweighted outliers
    unweighted_outliers = detect_outliers_iqr(dataset_df, 'Unweighted MSE')
    # Weighted outliers
    weighted_outliers = detect_outliers_iqr(dataset_df, 'Weighted MSE')
    
    print(f"\n{dataset} Set Outliers:")
    print(f"Unweighted MSE outliers: {unweighted_outliers.sum()}")
    print(f"Weighted MSE outliers: {weighted_outliers.sum()}")
    
    if unweighted_outliers.sum() > 0:
        print(f"\nUnweighted outliers in {dataset}:")
        print(dataset_df[unweighted_outliers][['Subject', 'Label', 'Unweighted MSE', 'Weighted MSE']])
    
    if weighted_outliers.sum() > 0:
        print(f"\nWeighted outliers in {dataset}:")
        print(dataset_df[weighted_outliers][['Subject', 'Label', 'Unweighted MSE', 'Weighted MSE']])

## Visualization - MSE Distribution

In [ ]:
# Box plots for MSE comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Validation - Unweighted
val_data = combined_df[combined_df['Dataset'] == 'Validation']
ax = axes[0, 0]
sns.boxplot(data=val_data, x='Label', y='Unweighted MSE', ax=ax)
ax.set_title('Validation Set - Unweighted MSE', fontsize=14, fontweight='bold')
ax.set_ylabel('MSE', fontsize=12)

# Validation - Weighted
ax = axes[0, 1]
sns.boxplot(data=val_data, x='Label', y='Weighted MSE', ax=ax)
ax.set_title('Validation Set - Weighted MSE', fontsize=14, fontweight='bold')
ax.set_ylabel('MSE', fontsize=12)

# Test - Unweighted
test_data = combined_df[combined_df['Dataset'] == 'Test']
ax = axes[1, 0]
sns.boxplot(data=test_data, x='Label', y='Unweighted MSE', ax=ax)
ax.set_title('Test Set - Unweighted MSE', fontsize=14, fontweight='bold')
ax.set_ylabel('MSE', fontsize=12)

# Test - Weighted
ax = axes[1, 1]
sns.boxplot(data=test_data, x='Label', y='Weighted MSE', ax=ax)
ax.set_title('Test Set - Weighted MSE', fontsize=14, fontweight='bold')
ax.set_ylabel('MSE', fontsize=12)

plt.tight_layout()
plt.savefig(os.path.join(mse_output_dir, 'mse_boxplots.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved boxplot to {os.path.join(mse_output_dir, 'mse_boxplots.png')}")

## Visualization - MSE Scatter Plot

In [ ]:
# Scatter plot: Unweighted vs Weighted MSE
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Validation
ax = axes[0]
val_data = combined_df[combined_df['Dataset'] == 'Validation']
for label in val_data['Label'].unique():
    label_data = val_data[val_data['Label'] == label]
    ax.scatter(label_data['Unweighted MSE'], label_data['Weighted MSE'], label=label, s=100, alpha=0.6)

# Add diagonal line
min_val = min(val_data['Unweighted MSE'].min(), val_data['Weighted MSE'].min())
max_val = max(val_data['Unweighted MSE'].max(), val_data['Weighted MSE'].max())
ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, label='Equal MSE')

ax.set_xlabel('Unweighted MSE', fontsize=12, fontweight='bold')
ax.set_ylabel('Weighted MSE', fontsize=12, fontweight='bold')
ax.set_title('Validation Set - Unweighted vs Weighted MSE', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Test
ax = axes[1]
test_data = combined_df[combined_df['Dataset'] == 'Test']
for label in test_data['Label'].unique():
    label_data = test_data[test_data['Label'] == label]
    ax.scatter(label_data['Unweighted MSE'], label_data['Weighted MSE'], label=label, s=100, alpha=0.6)

# Add diagonal line
min_val = min(test_data['Unweighted MSE'].min(), test_data['Weighted MSE'].min())
max_val = max(test_data['Unweighted MSE'].max(), test_data['Weighted MSE'].max())
ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, label='Equal MSE')

ax.set_xlabel('Unweighted MSE', fontsize=12, fontweight='bold')
ax.set_ylabel('Weighted MSE', fontsize=12, fontweight='bold')
ax.set_title('Test Set - Unweighted vs Weighted MSE', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(mse_output_dir, 'mse_scatter.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved scatter plot to {os.path.join(mse_output_dir, 'mse_scatter.png')}")

## Statistical Summary

In [ ]:
# Summary statistics
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

for dataset in ['Validation', 'Test']:
    print(f"\n{dataset.upper()} SET:")
    dataset_df = combined_df[combined_df['Dataset'] == dataset]
    
    print(f"\nTotal subjects: {len(dataset_df)}")
    print(f"Label distribution:")
    print(dataset_df['Label'].value_counts())
    
    print(f"\nUnweighted MSE:")
    print(f"  Mean: {dataset_df['Unweighted MSE'].mean():.6f}")
    print(f"  Std:  {dataset_df['Unweighted MSE'].std():.6f}")
    print(f"  Min:  {dataset_df['Unweighted MSE'].min():.6f}")
    print(f"  Max:  {dataset_df['Unweighted MSE'].max():.6f}")
    
    print(f"\nWeighted MSE:")
    print(f"  Mean: {dataset_df['Weighted MSE'].mean():.6f}")
    print(f"  Std:  {dataset_df['Weighted MSE'].std():.6f}")
    print(f"  Min:  {dataset_df['Weighted MSE'].min():.6f}")
    print(f"  Max:  {dataset_df['Weighted MSE'].max():.6f}")
    
    print(f"\nMSE Difference (Weighted - Unweighted):")
    print(f"  Mean: {dataset_df['MSE Difference'].mean():.6f}")
    print(f"  Std:  {dataset_df['MSE Difference'].std():.6f}")
    print(f"  Min:  {dataset_df['MSE Difference'].min():.6f}")
    print(f"  Max:  {dataset_df['MSE Difference'].max():.6f}")